###  GOLD LAYER - Business Insights & Analytics 

###  Databricks Notebook: Gold_Star_Schema 

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window 


credential = "banking_storage_credential"
storage = "bankingstorage26"

### 1. Create Dim_Customers Table (SCD Type 2)

In [0]:

# Read Silver tables 
customers_silver = spark.table("banking_catalog.silver.customers") 

accounts_silver = spark.table("banking_catalog.silver.accounts") 

transactions_silver = spark.table("banking_catalog.silver.transactions") 

loans_silver = spark.table("banking_catalog.silver.loans") 

fraud_silver = spark.table("banking_catalog.silver.fraud_alerts") 

atm_silver = spark.table("banking_catalog.silver.atm_transactions") 

branches_silver = spark.table("banking_catalog.silver.branches") 

credit_cards_silver = spark.table("banking_catalog.silver.credit_cards") 

employees_silver = spark.table("banking_catalog.silver.employees") 

kyc_silver = spark.table("banking_catalog.silver.kyc_documents") 



In [0]:
# Create Dim_Customers with SCD Type 2 
dim_customers = customers_silver \
    .withColumn("customer_sk", monotonically_increasing_id()) \
    .withColumn("effective_date", current_date()) \
    .withColumn("end_date", lit("9999-12-31").cast("date")) \
    .withColumn("is_current", lit(True)) \
    .withColumn("full_name", concat(col("name"), lit(" (ID: "),col("customer_id"), lit(")"))) \
    .withColumn("age", datediff(current_date(), col("dob")) / 365.25) \
    .withColumn("age_group",  when(col("age") < 30, "Young") 
    .when(col("age") < 50, "Middle") 
    .otherwise("Senior")) 
 
# Write to Gold 
dim_customers.write.format("delta") \
        .mode("overwrite") \
        .save(f"abfss://gold@{storage}.dfs.core.windows.net/dim_customers") 
print(f"✅Dim_Customers: {dim_customers.count()} records")

### 2. Create Dim_Accounts Table 

In [0]:
# Create Dim_Accounts 
dim_accounts = accounts_silver \
                .withColumn("account_sk", monotonically_increasing_id()) \
                .withColumn("effective_date", current_date()) \
                .withColumn("is_current", lit(True)) \
                .withColumn("account_age_days", datediff(current_date(), col("date"))) \
                .withColumn("balance_category", when(col("amount") < 10000, "Low") 
                .when(col("amount") < 50000, "Medium") .otherwise("High")) 
# Write to Gold 
dim_accounts.write.format("delta") \
        .mode("overwrite") \
        .save(f"abfss://gold@{storage}.dfs.core.windows.net/dim_accounts") 
print(f"✅Dim_Accounts: {dim_accounts.count()} records")

### 3.  Create Dim_Date Table

In [0]:
# Create Date Dimension (2020-2025) 
def create_date_dimension(start_date, end_date): 
    dates = spark.sql(f""" 
            SELECT EXPLODE(SEQUENCE( 
                TO_DATE('{start_date}'),  
                TO_DATE('{end_date}'),  
                INTERVAL 1 DAY 
            )) AS date 
        """) 
        
    date_dim = dates.select( 
            col("date"), 
            year("date").alias("year"), 
            quarter("date").alias("quarter"), 
            month("date").alias("month"), 
            dayofmonth("date").alias("day"), 
            dayofweek("date").alias("day_of_week"), 
            date_format("date", "EEEE").alias("day_name"), 
            date_format("date", "MMMM").alias("month_name"), 
            when(dayofweek("date").isin([1, 7]), "Weekend").otherwise("Weekday").alias("weekday_indicator")
        ).withColumn("date_sk", monotonically_increasing_id()) 

    return date_dim

date_dim_df = create_date_dimension("2020-01-01", "2025-12-31") 

date_dim_df.write.format("delta") \
        .mode("overwrite") \
        .save(f"abfss://gold@{storage}.dfs.core.windows.net/dim_date") 

print(f"✅Dim_Date: {date_dim_df.count()} records") 

### 4.  Create Fact_Transactions 

In [0]:
# Create Fact_Transactions with surrogate keys 
fact_transactions = transactions_silver \
                    .join(dim_accounts.select("account_id", "account_sk"), "account_id", "left") \
                    .join(dim_customers.select("customer_id", "customer_sk"), "customer_id", "left") \
                    .join(date_dim_df.select("date", "date_sk"), transactions_silver["date"] == date_dim_df["date"], "left") \
                    .select( 
                        "transaction_id", 
                        "customer_sk", 
                        "account_sk", 
                        "date_sk", 
                        "amount", 
                        "status", 
                        col("type").alias("transaction_type"),
                        "branch", 
                        "remarks", 
                        "flag", 
                        "year", 
                        "month", 
                        "quarter" 
                    ) 
 
# Write to Gold 
fact_transactions.write.format("delta") \
    .mode("overwrite") \
    .save(f"abfss://gold@{storage}.dfs.core.windows.net/fact_transactions") 
 
print(f"✅Fact_Transactions: {fact_transactions.count()} records")

### 5.  Create Fact_Loans 


In [0]:
# Create Fact_Loans 
fact_loans = loans_silver \
                .join(dim_accounts.select("account_id", "account_sk"), "account_id", "left") \
                .join(dim_customers.select("customer_id", "customer_sk"), "customer_id", "left") \
                .join(date_dim_df.select("date", "date_sk"), loans_silver["date"] == date_dim_df["date"], "left") \
                .select( 
                    "loan_id", 
                    "customer_sk", 
                    "account_sk", 
                    "date_sk", 
                    "amount", 
                    "status", 
                    col("type").alias("loan_type"), 
                    "branch", 
                    "remarks", 
                    "flag" 
                ) 
# Write to Gold 
fact_loans.write.format("delta") \
        .mode("overwrite") \
        .save(f"abfss://gold@{storage}.dfs.core.windows.net/fact_loans") 
print(f"✅Fact_Loans: {fact_loans.count()} records")

### 6. Create Fact_Fraud_Alerts

In [0]:
# Create Fact_Fraud_Alerts 
fact_fraud = fraud_silver \
                .join(dim_customers.select("customer_id", "customer_sk"), "customer_id", "left") \
                .join(dim_accounts.select("account_id", "account_sk"), "account_id", "left") \
                .join(date_dim_df.select("date", "date_sk"), fraud_silver["date"] == date_dim_df["date"], "left") \
                .select( 
                    "alert_id", 
                    "customer_sk", 
                    "account_sk", 
                    "date_sk", 
                    "amount", 
                    "status", 
                    col("type").alias("alert_type"), 
                            "branch", 
                            "remarks", 
                            "flag", 
                            "is_high_risk" 
                    ) 
 
# Write to Gold 
fact_fraud.write.format("delta") \
    .mode("overwrite") \
    .save(f"abfss://gold@{storage}.dfs.core.windows.net/fact_fraud_alerts") 
print(f"✅Fact_Fraud_Alerts: {fact_fraud.count()} records") 

### ==============================================

##  Create Gold Tables in Unity Catalog

In [0]:
%sql 
-- Create Gold tables 
CREATE TABLE IF NOT EXISTS banking_catalog.gold.dim_customers 
USING DELTA 
LOCATION 'abfss://gold@bankingstorage26.dfs.core.windows.net/dim_customers/'; 

CREATE TABLE IF NOT EXISTS banking_catalog.gold.dim_accounts 
USING DELTA 
LOCATION 'abfss://gold@bankingstorage26.dfs.core.windows.net/dim_accounts/'; 

CREATE TABLE IF NOT EXISTS banking_catalog.gold.dim_date 
USING DELTA 
LOCATION 'abfss://gold@bankingstorage26.dfs.core.windows.net/dim_date/'; 

CREATE TABLE IF NOT EXISTS banking_catalog.gold.fact_transactions 
USING DELTA 
LOCATION 'abfss://gold@bankingstorage26.dfs.core.windows.net/fact_transactions/'; 

CREATE TABLE IF NOT EXISTS banking_catalog.gold.fact_loans 
USING DELTA 
LOCATION 'abfss://gold@bankingstorage26.dfs.core.windows.net/fact_loans/'; 

CREATE TABLE IF NOT EXISTS banking_catalog.gold.fact_fraud_alerts 
USING DELTA 
LOCATION 'abfss://gold@bankingstorage26.dfs.core.windows.net/fact_fraud_alerts/';

In [0]:
%sql
-- DROP TABLE IF EXISTS banking_catalog.gold.dim_customers;
-- DROP TABLE IF EXISTS banking_catalog.gold.dim_accounts;
-- DROP TABLE IF EXISTS banking_catalog.gold.dim_date;
-- DROP TABLE IF EXISTS banking_catalog.gold.fact_transactions;
-- DROP TABLE IF EXISTS banking_catalog.gold.fact_loans;
-- DROP TABLE IF EXISTS banking_catalog.gold.fact_fraud_alerts;
